# CarbonGlobe — LSTM-XXL (much larger DeepED)

**IEEE Big Data Cup 2026 — CarbonGlobe AI Emulation Challenge**

Scales the winning DeepED design up: a **4-layer bidirectional LSTM encoder at d=512** with residual +
LayerNorm + stochastic depth, and **7 per-variable branches at 448 units** with deeper MLP heads
(≈2-3× the parameters of LSTM-XL). Keeps the state-aware fused input and the residual-on-previous-state
that made the smaller version reach LB 0.168.

Same training recipe as the LB-0.149 flagship: scheduled sampling, EMA, multi-seed, clamp, early stop.

## 0. Clone repo & imports

In [ ]:
import os, sys, subprocess, time, gc

REPO_DIR = '/kaggle/working/carbon-globe'
REPO_URL = 'github.com/kuongan/carbon-globe.git'

def _github_token():
    # Store the token in a Kaggle Secret named GITHUB_TOKEN (Add-ons -> Secrets).
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        return os.environ.get('GITHUB_TOKEN', '')

if not os.path.isdir(REPO_DIR):
    tok = _github_token()
    url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr.replace(tok, '***') if tok else r.stderr
        raise RuntimeError('git clone failed:\n' + err)

PKG_DIR = os.path.join(REPO_DIR, 'LSTM-LSTNet-DeepED')
sys.path.insert(0, PKG_DIR)
os.chdir(PKG_DIR)
print('cwd =', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm

from configs.constant_glob import N_CONS, N_OUT, N_YEAR, N_T, N_FEA, TREE_BAND
from utils.func import normData, invNormData
from models.model import MM_LSTM_Age_v2   # param-count reference

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| Keras', keras.__version__)
print('N_CONS=%d N_OUT=%d N_FEA=%d N_YEAR=%d N_T=%d' % (N_CONS, N_OUT, N_FEA, N_YEAR, N_T))
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print('memory growth note:', e)

## 1. Constants, stats & competition metric

In [ ]:
COMP = '/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public'
BASE = '/kaggle/input/datasets/zhihaow/carbonglobe'
OUT  = '/kaggle/working'

SEED_AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
SCENARIOS = ['ssp126', 'ssp585']
N_AGES, N_AGE_FEA = len(SEED_AGES), 15

stats = np.load(f'{BASE}/data_stats/data_stats.npz')
X_MEAN, X_STD = stats['x_mean'].astype(np.float32), stats['x_std'].astype(np.float32)
Y_MEAN, Y_STD = stats['y_mean'].astype(np.float32), stats['y_std'].astype(np.float32)

agetri = np.load(f'{BASE}/data_stats/age_triplet.npz')
TRI_AGE   = agetri['age'].astype(np.float32)
TRI_MEAN  = agetri['age_mean'].astype(np.float32)
TRI_SLOPE = agetri['age_slope'].astype(np.float32)
assert list(TRI_AGE.astype(int)) == SEED_AGES

AGE_MEAN_N  = normData(TRI_MEAN,  Y_MEAN, Y_STD)
AGE_SLOPE_N = normData(TRI_SLOPE, Y_MEAN, Y_STD)
AGE_FEAT = np.stack([np.concatenate([[TRI_AGE[i] / 500.0], AGE_MEAN_N[i], AGE_SLOPE_N[i]])
                     for i in range(N_AGES)]).astype(np.float32)
FLOOR_N = normData(np.zeros(N_OUT, np.float32), Y_MEAN, Y_STD).astype(np.float32)
print('Y_MEAN =', np.round(Y_MEAN, 5))
print('submission scale -> height /%.5f , agb /%.5f' % (Y_MEAN[0], Y_MEAN[1]))


def scored_metric(pred, true):
    """The competition metric on year-40 height & AGB (physical units in, scaled MSE out)."""
    ph, th = pred[..., 0] / Y_MEAN[0], true[..., 0] / Y_MEAN[0]
    pa, ta = pred[..., 1] / Y_MEAN[1], true[..., 1] / Y_MEAN[1]
    mse_h = float(np.mean((ph - th) ** 2)); mse_a = float(np.mean((pa - ta) ** 2))
    return 0.5 * (mse_h + mse_a), mse_h, mse_a

## 2. Load train / val arrays (standardized in place)

In [ ]:
t0 = time.time()
res = np.load(f'{BASE}/data_global/res_train4_test8.npz', mmap_mode='r')
x_train = np.asarray(res['x_train'], dtype=np.float32)
x_val   = np.asarray(res['x_test'],  dtype=np.float32)
y_train = np.ascontiguousarray(res['y_train'][..., 11, :]).astype(np.float32)
y_val   = np.ascontiguousarray(res['y_test'][...,  11, :]).astype(np.float32)
res.close(); gc.collect()

def standardize_inplace(x):
    if not x.flags.writeable: x = x.copy()
    x -= X_MEAN; x /= (X_STD + 1e-10)
    return x

x_train   = standardize_inplace(x_train)
x_val     = standardize_inplace(x_val)
y_train_n = normData(y_train, Y_MEAN, Y_STD).astype(np.float32)
del y_train; gc.collect()

val_y0   = y_val[:, :, 0, :]
val_true = y_val[:, :, 40, :]
N_TRAIN  = x_train.shape[0]
N_PAIRS  = N_AGES * N_YEAR * N_TRAIN
print('loaded in %.1fs | x_train' % (time.time() - t0), x_train.shape, '| x_val', x_val.shape)

## 3. Model — LSTM-XXL

In [ ]:
# ================= LSTM-XXL: a much larger BiLSTM DeepED =================
# Same winning recipe as LSTM-XL (state-aware fused input, residual per-variable branches) but
# 4-layer BiLSTM encoder at d=512, 448-unit branches, deeper MLP heads and stochastic depth.

class ResBiLSTM(layers.Layer):
    def __init__(self, d_model, dropout, drop_path=0.0, **kw):
        super().__init__(**kw)
        self.lstm = layers.Bidirectional(layers.LSTM(d_model // 2, return_sequences=True))
        self.norm = layers.LayerNormalization(epsilon=1e-5)
        self.drop = layers.Dropout(dropout)
        self.drop_path = drop_path
    def call(self, x, training=False):
        y = self.drop(self.lstm(x), training=training)
        if training and self.drop_path > 0:
            keep = 1.0 - self.drop_path
            mask = tf.floor(keep + tf.random.uniform((tf.shape(x)[0], 1, 1)))
            return self.norm(x + y * mask / keep)
        return self.norm(x + y)


class BigBranch(layers.Layer):
    def __init__(self, idx, units, mlp_units, dropout, **kw):
        super().__init__(name=f'branch{idx}', **kw)
        self.idx = idx
        self.lstm_data = layers.LSTM(units, return_sequences=True)
        self.lstm_gate = layers.LSTM(units, return_sequences=True)
        self.gate      = layers.Dense(units, activation='sigmoid')
        self.lstm_out  = layers.LSTM(units, return_sequences=True)
        self.norm      = layers.LayerNormalization(epsilon=1e-5)
        self.pool      = layers.Dense(1)
        self.mlp1      = layers.Dense(mlp_units, activation='gelu')
        self.mlp2      = layers.Dense(mlp_units // 2, activation='gelu')
        self.mlp3      = layers.Dense(128, activation='gelu')
        self.drop      = layers.Dropout(dropout)
        self.head      = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros')
    def call(self, h, prev, age, training=False):
        i = self.idx; T = tf.shape(h)[1]
        age_i = tf.concat([age[:, :1], age[:, i + 1:i + 2], age[:, i + 8:i + 9]], axis=-1)
        loc = tf.concat([prev[:, i:i + 1], age_i], axis=-1)
        z = tf.concat([h, tf.repeat(loc[:, None, :], T, axis=1)], axis=-1)
        s = self.lstm_data(z) * self.gate(self.lstm_gate(z))
        s = self.norm(self.lstm_out(s))
        w = tf.nn.softmax(self.pool(s), axis=1)
        feat = tf.concat([tf.reduce_sum(w * s, axis=1), s[:, -1]], axis=-1)
        feat = self.drop(self.mlp1(feat), training=training)
        feat = self.mlp3(self.mlp2(feat))
        feat = tf.concat([feat, prev, age_i], axis=-1)
        return prev[:, i:i + 1] + self.head(feat)


class LSTM_XXL(keras.Model):
    def __init__(self, n_out=N_OUT, n_cons=N_CONS, n_age_fea=N_AGE_FEA, d_model=512, n_shared=4,
                 branch_units=448, mlp_units=768, dropout=0.1, drop_path=0.1, **kw):
        super().__init__(**kw)
        self.n_out, self.n_cons, self.n_age_fea = n_out, n_cons, n_age_fea
        self.ctx_proj = layers.Dense(160, activation='gelu')
        self.in_proj  = layers.Dense(d_model)
        self.in_norm  = layers.LayerNormalization(epsilon=1e-5)
        self.encoder  = [ResBiLSTM(d_model, dropout, drop_path * i / max(n_shared - 1, 1)) for i in range(n_shared)]
        self.branches = [BigBranch(i, branch_units, mlp_units, dropout) for i in range(n_out)]
    def call(self, inputs, training=False):
        nc, no, na = self.n_cons, self.n_out, self.n_age_fea
        env = inputs[:, :, :nc]; prev = inputs[:, -1, nc:nc + no]; age = inputs[:, -1, nc + no:nc + no + na]
        T = tf.shape(env)[1]
        ctx = self.ctx_proj(tf.concat([prev, age], axis=-1))
        h = tf.concat([env, tf.repeat(ctx[:, None, :], T, axis=1)], axis=-1)
        h = self.in_norm(self.in_proj(h))
        for blk in self.encoder:
            h = blk(h, training=training)
        return tf.concat([b(h, prev, age, training=training) for b in self.branches], axis=-1)

## 4. Config

In [ ]:
RUN_NAME = 'lstm_xxl'

# ---- optimisation ----
BATCH_SIZE   = 384
ROLL_TOKENS  = 1280
MIN_BATCH    = 48
BASE_LR      = 4e-4
MIN_LR       = 1e-5
WARMUP_STEPS = 1500
CLIP_NORM    = 1.0
WEIGHT_DECAY = 1e-4
Y_W          = [2., 2., 1., 1., 1., 1., 1.]
CLAMP_NONNEG = True

# ---- epochs / early stop / LR plateau ----
MAX_EPOCHS   = 24
SAMPLES_PER_EPOCH = 1_000_000
LR_FACTOR, LR_PATIENCE = 0.5, 2
ES_PATIENCE  = 5
MIN_DELTA    = 1e-4
TIME_BUDGET_H = 8.5

# ---- scheduled-sampling curriculum: (start_epoch, K years/window, teacher-forcing prob) ----
CURRICULUM = [(1, 1, 1.00), (3, 4, 0.75), (6, 8, 0.50), (9, 12, 0.25), (12, 16, 0.0)]

# ---- EMA of weights (evaluated each epoch) ----
USE_EMA   = True
EMA_DECAY = 0.999

# ---- multi-seed averaging ----
N_SEEDS = 1

def make_model():
    return LSTM_XXL(d_model=512, n_shared=4, branch_units=448, mlp_units=768, dropout=0.1, drop_path=0.1)

def make_optimizer(lr):
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY)
    try:
        opt.exclude_from_weight_decay(var_names=['bias', 'gamma', 'beta', 'layer_normalization', 'embeddings'])
    except Exception as e:
        print('exclude_from_weight_decay note:', e)
    return opt

def warm_start(model):
    pass   # single-backbone models train from scratch; set INIT via a Kaggle dataset if desired

print('model:', RUN_NAME, '| curriculum', CURRICULUM, '| EMA', USE_EMA, '| seeds', N_SEEDS)

## 5. Training engine (window sampler, rollout-aware step, EMA, rollout, LR & early stop)

Identical to the proven engine in the single-model notebooks, plus an EMA shadow of the weights that is
swapped in for each epoch's validation rollout.

In [ ]:
FLOOR_N = normData(np.zeros(N_OUT, np.float32), Y_MEAN, Y_STD).astype(np.float32)
FLOOR_T = tf.constant(FLOOR_N); Y_W_T = tf.constant(Y_W, tf.float32)

def batch_size_for(K):
    return int(min(BATCH_SIZE, max(MIN_BATCH, ROLL_TOKENS // K)))

def window_batches(K, n_windows, batch, rng):
    T0 = N_YEAR - K + 1
    total = N_AGES * N_TRAIN * T0
    n = min(n_windows, total)
    idx = rng.choice(total, size=n, replace=False)
    kx, ky = np.arange(K), np.arange(K + 1)
    for s in range(0, n - batch + 1, batch):
        b = idx[s:s + batch]
        ai, rem = np.divmod(b, N_TRAIN * T0)
        site, t0 = np.divmod(rem, T0)
        env = x_train[site[:, None], t0[:, None] + kx]
        yy  = y_train_n[ai[:, None], site[:, None], t0[:, None] + ky]
        yield env, yy, AGE_FEAT[ai]

def make_dataset(K, n_windows, batch, rng):
    sig = (tf.TensorSpec((batch, K, N_T, N_CONS), tf.float32),
           tf.TensorSpec((batch, K + 1, N_OUT), tf.float32),
           tf.TensorSpec((batch, N_AGE_FEA), tf.float32))
    return (tf.data.Dataset.from_generator(lambda: window_batches(K, n_windows, batch, rng),
                                           output_signature=sig).prefetch(8))

def make_train_step(model, optimizer, ema):
    @tf.function
    def train_step(env, yy, af, tf_ratio):
        K = env.shape[1]; B = tf.shape(env)[0]
        af_m = tf.repeat(af[:, None, :], N_T, axis=1)
        with tf.GradientTape() as tape:
            prev = yy[:, 0]; loss = 0.0
            for k in range(K):
                prev_m = tf.repeat(prev[:, None, :], N_T, axis=1)
                xk = tf.concat([env[:, k], prev_m, af_m], axis=-1)
                pred = model(xk, training=True)
                if CLAMP_NONNEG: pred = tf.maximum(pred, FLOOR_T)
                loss += tf.reduce_mean(Y_W_T * tf.square(pred - yy[:, k + 1]))
                if k < K - 1:
                    keep = tf.cast(tf.random.uniform((B, 1)) < tf_ratio, tf.float32)
                    prev = keep * yy[:, k + 1] + (1.0 - keep) * pred
            loss = loss / tf.cast(K, tf.float32)
        grads = tape.gradient(loss, model.trainable_variables)
        grads, gnorm = tf.clip_by_global_norm(grads, CLIP_NORM)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        if ema is not None:
            for e, v in zip(ema, model.trainable_variables):
                e.assign(EMA_DECAY * e + (1.0 - EMA_DECAY) * v)
        return loss, gnorm
    return train_step

def rollout(model, x, y0_raw, x_is_norm=False, desc=None, bs=8192):
    N, A = x.shape[0], y0_raw.shape[0]
    prev = normData(y0_raw, Y_MEAN, Y_STD).reshape(A * N, N_OUT).astype(np.float32)
    site_idx = np.tile(np.arange(N), A)
    af_rows = AGE_FEAT[np.repeat(np.arange(A), N)]
    @tf.function(reduce_retracing=True)
    def infer(xb): return model(xb, training=False)
    years = tqdm(range(N_YEAR), desc=desc, leave=False) if desc else range(N_YEAR)
    for t in years:
        env_t = x[:, t] if x_is_norm else normData(x[:, t], X_MEAN, X_STD).astype(np.float32)
        nxt = np.empty_like(prev)
        for s in range(0, A * N, bs):
            e = min(s + bs, A * N)
            xb = np.concatenate([env_t[site_idx[s:e]],
                                 np.repeat(prev[s:e, None, :], N_T, axis=1),
                                 np.repeat(af_rows[s:e, None, :], N_T, axis=1)], axis=-1)
            p = infer(tf.constant(xb, tf.float32)).numpy()
            nxt[s:e] = np.maximum(p, FLOOR_N) if CLAMP_NONNEG else p
        prev = nxt
    out = invNormData(prev, Y_MEAN, Y_STD).reshape(A, N, N_OUT)
    return np.maximum(out, 0.0) if CLAMP_NONNEG else out

def set_lr(opt, v): opt.learning_rate = float(v)

class LRController:
    def __init__(self, opt, base_lr, warmup, factor, patience, min_lr, min_delta):
        self.opt, self.base_lr, self.warmup = opt, base_lr, warmup
        self.factor, self.patience, self.min_lr, self.min_delta = factor, patience, min_lr, min_delta
        self.step, self.best, self.wait = 0, np.inf, 0
        self.lr = base_lr * (0.1 if warmup > 0 else 1.0); set_lr(opt, self.lr)
    def on_step(self):
        if self.step < self.warmup:
            self.lr = self.base_lr * (0.1 + 0.9 * (self.step + 1) / self.warmup); set_lr(self.opt, self.lr)
        self.step += 1
    def on_epoch(self, metric, counting=True):
        if metric < self.best - self.min_delta: self.best, self.wait = metric, 0
        elif counting and self.step >= self.warmup:
            self.wait += 1
            if self.wait >= self.patience and self.lr > self.min_lr:
                self.lr = max(self.lr * self.factor, self.min_lr); set_lr(self.opt, self.lr); self.wait = 0
                print(f'    ReduceLROnPlateau -> lr = {self.lr:.2e}')

def stage_for_epoch(ep):
    si = 0
    for j, (start, _, _) in enumerate(CURRICULUM):
        if ep >= start: si = j
    _, K, tfr = CURRICULUM[si]
    return si, K, tfr

print('engine ready | batch per K:', {K: batch_size_for(K) for _, K, _ in CURRICULUM})

## 6. Train (multi-seed × scheduled sampling, EMA, early stop, LR schedule)

In [ ]:
BACKBONE = RUN_NAME  # single-backbone notebook
LSTM_INIT = ECO_INIT = ''
def train_one(seed):
    tf.keras.utils.set_random_seed(seed)
    model = make_model()
    _ = model(tf.zeros((2, N_T, N_FEA), tf.float32), training=False)   # build
    warm_start(model)
    if seed == SEED0:
        ref = MM_LSTM_Age_v2(n_out=N_OUT, n_cons=N_CONS)
        _ = ref(tf.zeros((2, N_T, N_FEA), tf.float32), training=False)
        print(f'{RUN_NAME}/{BACKBONE} seed {seed}: {model.count_params()/1e6:.2f}M params '
              f'(v2 ref {ref.count_params()/1e6:.2f}M)')
        del ref; gc.collect()

    optimizer = make_optimizer(BASE_LR)
    optimizer.build(model.trainable_variables)
    ema = [tf.Variable(v, trainable=False) for v in model.trainable_variables] if USE_EMA else None
    train_step = make_train_step(model, optimizer, ema)
    lr_ctl = LRController(optimizer, BASE_LR, WARMUP_STEPS, LR_FACTOR, LR_PATIENCE, MIN_LR, MIN_DELTA)

    rng = np.random.default_rng(seed)
    best = {'score': np.inf, 'weights': None, 'epoch': 0, 'src': 'raw'}
    wait, last_stage, t_start = 0, None, time.time()
    spe = SAMPLES_PER_EPOCH or N_PAIRS

    for ep in range(1, MAX_EPOCHS + 1):
        t_ep = time.time(); si, K, tfr = stage_for_epoch(ep)
        counting = (si == len(CURRICULUM) - 1)
        if si != last_stage:
            if last_stage is not None: wait, lr_ctl.wait = 0, 0
            print(f'  == stage {si}: K={K}, teacher-forcing={tfr:.2f}'); last_stage = si
        B = batch_size_for(K); n_win = max(B, spe // K); steps = n_win // B
        tfr_t = tf.constant(tfr, tf.float32)
        running, nb = 0.0, 0
        pbar = tqdm(make_dataset(K, n_win, B, rng), total=steps, desc=f'seed{seed} ep{ep}/{MAX_EPOCHS} K={K}')
        for env, yy, af in pbar:
            lr_ctl.on_step()
            loss, _ = train_step(env, yy, af, tfr_t)
            running += float(loss); nb += 1
            if nb % 50 == 0 or nb == steps: pbar.set_postfix(loss=f'{running/max(nb,1):.5f}', lr=f'{lr_ctl.lr:.1e}')
        train_loss = running / max(nb, 1)

        # snapshot the live (raw) weights so training always continues from them
        raw_w = [v.numpy() for v in model.trainable_variables]
        vpred = rollout(model, x_val, val_y0, x_is_norm=True)
        vscore, vh, va = scored_metric(vpred, val_true)
        cand = {'score': vscore, 'weights': raw_w, 'src': 'raw', 'h': vh, 'a': va}

        if ema is not None:   # evaluate the EMA shadow without disturbing the raw path
            for v, e in zip(model.trainable_variables, ema): v.assign(e)
            epred = rollout(model, x_val, val_y0, x_is_norm=True)
            escore, eh, ea = scored_metric(epred, val_true)
            for v, w in zip(model.trainable_variables, raw_w): v.assign(w)   # always restore raw
            if escore < cand['score']:
                cand = {'score': escore, 'weights': [e.numpy() for e in ema], 'src': 'ema', 'h': eh, 'a': ea}

        vscore, vh, va, tag = cand['score'], cand['h'], cand['a'], cand['src']
        improved = vscore < best['score'] - MIN_DELTA
        if vscore < best['score']:
            best = {'score': vscore, 'weights': cand['weights'], 'epoch': ep, 'src': tag}

        lr_ctl.on_epoch(vscore, counting)
        if counting: wait = 0 if improved else wait + 1
        ep_min = (time.time() - t_ep) / 60
        print(f'  seed{seed} ep{ep}: train {train_loss:.5f} | VAL {vscore:.5f} ({tag}; h {vh:.4f}, agb {va:.4f}) '
              f'| lr {lr_ctl.lr:.1e} | {ep_min:.1f}m' + ('  *best*' if best['epoch'] == ep else ''))
        if counting and wait >= ES_PATIENCE:
            print(f'  early stop (best ep {best["epoch"]}, VAL {best["score"]:.5f})'); break
        if (time.time() - t_start) / 3600 + ep_min / 60 > TIME_BUDGET_H:
            print('  time budget reached -> stop'); break

    for v, w in zip(model.trainable_variables, best['weights']): v.assign(w)
    model.save_weights(f'{OUT}/{RUN_NAME}_seed{seed}.weights.h5')
    print(f'  seed {seed}: best VAL {best["score"]:.5f} ({best["src"]}, ep {best["epoch"]})')
    return model, best['score']

SEED0 = SEED
seed_models, seed_scores = [], []
for si in range(N_SEEDS):
    m, sc = train_one(SEED0 + si)
    seed_models.append(m); seed_scores.append(sc)
print('\nper-seed VAL:', [round(s, 5) for s in seed_scores])

## 7. Validate the seed-averaged model

In [ ]:
def rollout_avg(x, y0_raw, x_is_norm=False, desc=None):
    acc = None
    for m in seed_models:
        p = rollout(m, x, y0_raw, x_is_norm=x_is_norm, desc=desc)
        acc = p if acc is None else acc + p
    return acc / len(seed_models)

val_pred = rollout_avg(x_val, val_y0, x_is_norm=True, desc='val rollout')
score, mse_h, mse_a = scored_metric(val_pred, val_true)
print(f'ENSEMBLE-of-seeds VAL metric = {score:.5f}  (height {mse_h:.5f}, agb {mse_a:.5f})')
if len(seed_scores) > 1:
    print('best single seed VAL =', round(min(seed_scores), 5))

## 8. Predict on the test set & write the submission

In [ ]:
def load_test(scn):
    d = np.load(f'{COMP}/test_{scn}.npz')
    return d['test_x'].astype(np.float32), d['test_y0'].astype(np.float32)

sample_sub = pd.read_csv(f'{COMP}/sample_submission.csv')
parts, raw_preds = [], {}
for scn in SCENARIOS:
    tx, ty0 = load_test(scn)
    N = tx.shape[0]
    p = rollout_avg(tx, np.transpose(ty0, (1, 0, 2)), desc=f'{scn} rollout')
    raw_preds[scn] = p
    ids = [f'{scn}_{site}_{age}' for age in SEED_AGES for site in range(N)]
    parts.append(pd.DataFrame({'id': ids,
                               'height': (p[:, :, 0] / Y_MEAN[0]).reshape(-1),
                               'agb':    (p[:, :, 1] / Y_MEAN[1]).reshape(-1)}))
    del tx, ty0; gc.collect()

sub = sample_sub[['id']].join(pd.concat(parts).set_index('id'), on='id')
assert not sub[['height', 'agb']].isna().any().any(), 'NaNs / missing ids!'
sub.to_csv(f'{OUT}/submission.csv', index=False)
np.savez_compressed(f'{OUT}/{RUN_NAME}_preds.npz', val_pred=val_pred, **raw_preds)
print('wrote', f'{OUT}/submission.csv', 'and', f'{OUT}/{RUN_NAME}_preds.npz')
print(sub.head())
print('height scaled mean %.3f std %.3f | agb mean %.3f std %.3f'
      % (sub['height'].mean(), sub['height'].std(), sub['agb'].mean(), sub['agb'].std()))

## 9. Push further

* **Add `lstm_xxl_preds.npz` to the ensemble notebook** — a bigger blend almost always helps; this model is
  architecturally distinct from the LSTM/EcoFormer/fusion runs, so it adds real diversity.
* **Scale knobs** in the config: `d_model`, `n_blocks`/`n_shared`, `d_ff`, `branch_units`. Raise `N_SEEDS`
  for seed-averaging once one seed fits the session.
* **Warm-start**: attach a previous `*.weights.h5` as a dataset and load it in `warm_start` to fine-tune.